In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.set_num_threads(12)  # i5-13500H: 12 cores
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cpu")
print(f"Using device: {device}, threads: {torch.get_num_threads()}")

Using device: cpu, threads: 12


In [5]:
from scipy.signal import resample_poly
import os

PROCESSED_DIR = "../data/processed"   # your .npy files saved by batch_preprocessing.ipynb
DOWNSAMPLED_DIR = "../data/processed_ds"  # new folder one level up, next to data/processed/
os.makedirs(DOWNSAMPLED_DIR, exist_ok=True)

ORIGINAL_FS = 700
TARGET_FS = 100
DOWNSAMPLE_FACTOR = ORIGINAL_FS // TARGET_FS  # = 7

subjects = list(range(2, 18))
subjects.remove(12)  # S12 excluded in WESAD

for sid in subjects:
    X = np.load(f"{PROCESSED_DIR}/S{sid}_X.npy")  # (N, 4, 21000)
    y = np.load(f"{PROCESSED_DIR}/S{sid}_y.npy")  # (N,)

    N, n_mod, n_samples = X.shape
    new_len = n_samples // DOWNSAMPLE_FACTOR  # 3000

    X_ds = np.zeros((N, n_mod, new_len), dtype=np.float32)
    for i in range(N):
        for m in range(n_mod):
            X_ds[i, m] = resample_poly(X[i, m], up=1, down=DOWNSAMPLE_FACTOR)

    np.save(f"{DOWNSAMPLED_DIR}/S{sid}_X.npy", X_ds)
    np.save(f"{DOWNSAMPLED_DIR}/S{sid}_y.npy", y)
    print(f"S{sid}: {X_ds.shape}")

S2: (140, 4, 3000)
S3: (142, 4, 3000)
S4: (143, 4, 3000)
S5: (146, 4, 3000)
S6: (145, 4, 3000)
S7: (145, 4, 3000)
S8: (146, 4, 3000)
S9: (145, 4, 3000)
S10: (150, 4, 3000)
S11: (147, 4, 3000)
S13: (147, 4, 3000)
S14: (147, 4, 3000)
S15: (147, 4, 3000)
S16: (147, 4, 3000)
S17: (150, 4, 3000)


In [6]:
class ModalityEncoder(nn.Module):
    """Encodes a single-channel 1D signal (3000 samples) into a 256-dim embedding."""
    def __init__(self, embed_dim=256):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=7, stride=2, padding=3)
        self.bn1   = nn.BatchNorm1d(32)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, stride=2, padding=2)
        self.bn2   = nn.BatchNorm1d(64)
        self.conv3 = nn.Conv1d(64, 128, kernel_size=3, stride=2, padding=1)
        self.bn3   = nn.BatchNorm1d(128)
        self.pool  = nn.AdaptiveAvgPool1d(64)
        self.fc    = nn.Linear(128 * 64, embed_dim)

    def forward(self, x):
        # x: (batch, 1, 3000)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.pool(x)              # (batch, 128, 64)
        x = x.flatten(1)              # (batch, 128*64)
        x = self.fc(x)                # (batch, 256)
        return x

In [7]:
class MultiModalCNN(nn.Module):
    def __init__(self, n_modalities=4, embed_dim=256, n_classes=2):
        super().__init__()
        self.n_modalities = n_modalities
        self.encoders = nn.ModuleList([ModalityEncoder(embed_dim) for _ in range(n_modalities)])
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * n_modalities, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, n_classes)
        )

    def forward(self, x, missing_mask=None):
        # x: (batch, n_modalities, 3000)
        embeddings = []
        for i, encoder in enumerate(self.encoders):
            emb = encoder(x[:, i:i+1, :])
            if missing_mask is not None and missing_mask[i]:
                emb = torch.zeros_like(emb)
            embeddings.append(emb)
        fused = torch.cat(embeddings, dim=1)  # (batch, 1024)
        return self.classifier(fused)

In [8]:
model = MultiModalCNN(n_modalities=4, embed_dim=256, n_classes=2).to(device)
dummy = torch.randn(8, 4, 3000)  # batch of 8
out = model(dummy)
print(out.shape)  # should be (8, 2)

n_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {n_params:,}")

torch.Size([8, 2])
Total parameters: 8,811,458
